In [2]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

1. Embeddings

1.1 Simplest possible version: one hot vectors

In [3]:
vocab = ["good", "great", "bad", "shoe", "movie"]
one_hot = {word: np.eye(len(vocab))[i] for i, word in enumerate(vocab)}

for w, v in one_hot.items():
    print(w.ljust(6), v)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

print("\nsim(good, great):", cosine_sim(one_hot["good"], one_hot["great"]))    # 0.0 -- looks totally unrelated!
print("sim(good, shpe):", cosine_sim(one_hot["good"], one_hot["shoe"]))         #also 0.0 --same "distance"   

good   [1. 0. 0. 0. 0.]
great  [0. 1. 0. 0. 0.]
bad    [0. 0. 1. 0. 0.]
shoe   [0. 0. 0. 1. 0.]
movie  [0. 0. 0. 0. 1.]

sim(good, great): 0.0
sim(good, shpe): 0.0


1.2 A step up: TF-IDF vectors

In [5]:
# --- Example 1: TF-IDF on a few sentences, then cosine similarity ---

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "AlfredX is a PyQt5 based AI voice assistant.",
    "AlfredX supports voice commands and reminders.",
    "The stock market fell sharply today.",
    "Investors reacted to the market crash.",
]


vec = TfidfVectorizer()
tfidf_matrix = vec.fit_transform(docs)
print("Shape:", tfidf_matrix.shape )
print("Vacabulary Size:", len(vec.vocabulary_))

sims = cosine_similarity(tfidf_matrix)
print("\nSimilarity Matrix:\n", np.round(sims, 2))

Shape: (4, 21)
Vacabulary Size: 21

Similarity Matrix:
 [[1.   0.22 0.   0.  ]
 [0.22 1.   0.   0.  ]
 [0.   0.   1.   0.24]
 [0.   0.   0.24 1.  ]]


In [ ]:
# Example 2: embeded new Query and find closest document (this IS retrieval, in miniature) ---

query = "Tell me about the voice assistant"
query_vec = vec.transform([query])                # use the SAME fitted vectorizer -- don't refit on the query 

sims_to_query = cosine_similarity(query_vec, tfidf_matrix)[0]
for doc, score in sorted(zip(docs, sims_to_query), key= lambda x: -x[1]):
    print(f"{score:.3f} {doc}")

0.433, AlfredX is a PyQt5 based AI voice assistant.
0.181, AlfredX supports voice commands and reminders.
0.181, The stock market fell sharply today.
0.181, Investors reacted to the market crash.


In [8]:
# proof of the weakness

query2 = "Tell me about the AI companion app"
q2_vec = vec.transform([query2])

sims2 = cosine_similarity(q2_vec, tfidf_matrix)[0]
for doc, score in sorted(zip(docs, sims2), key= lambda x: -x[1]):
    print(f"{score:.3f} {doc}")

0.314 AlfredX is a PyQt5 based AI voice assistant.
0.213 The stock market fell sharply today.
0.213 Investors reacted to the market crash.
0.000 AlfredX supports voice commands and reminders.


1.3 Real dense based embeddings: train a tinyu one from scratch (offline to see what's actually hapening)

In [9]:
import torch, torch.nn as nn, torch.nn.functional as F

# Tiny vocabulary and "training pairs": words that tend to appear in similar contexts get pulled together
vocab2 = ["good", "great", "excellent", "bad", "terrible", "awful", "shoe", "sneaker", "boot"]
word2id = {w: i for i, w in enumerate(vocab2)}

# Positive pairs = should end up CLOSE in vector space (same "cluster")
positive_pairs = [("good","great"), ("great","excellent"), ("good","excellent"),
                  ("bad","terrible"), ("terrible","awful"), ("bad","awful"),
                  ("shoe","sneaker"), ("sneaker","boot"), ("shoe","boot")]


embed_dim = 8
embedding = nn.Embedding(len(vocab2), embed_dim)
optimizer = torch.optim.Adam(embedding.parameters(), lr=0.05)

for epoch in range(300):
    total_loss = 0
    for w1, w2 in positive_pairs:
        v1 = embedding(torch.tensor(word2id[w1]))
        v2 = embedding(torch.tensor(word2id[w2]))
        # pull positive pairs together: maximize cosine similarity -> minimize (1 - cos_sim)
        loss = 1 - F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)) 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 60 == 0:
        print(f"epoch {epoch}  loss {total_loss:.3f}")    


epoch 0  loss 8.837
epoch 60  loss 0.000
epoch 120  loss -0.000
epoch 180  loss 0.000
epoch 240  loss 0.002


In [12]:

# --- Now check: did semantically similar words end up with similar vectors, with NO shared characters? ---

def sim(w1, w2):
    v1 = embedding(torch.tensor(word2id[w1]))
    v2 = embedding(torch.tensor(word2id[w2]))
    return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0))

print("sim(good, great)      :", round(sim("good","great").item(), 3))       # trained to be close
print("sim(good, excellent)  :", round(sim("good","excellent").item(), 3))   # trained to be close, no shared letters!
print("sim(good, terrible)   :", round(sim("good","terrible").item(), 3))    # never linked -> should be lower
print("sim(good, shoe)       :", round(sim("good","shoe").item(), 3))        # different cluster entirely -> lowest
    

sim(good, great)      : 1.0
sim(good, excellent)  : 1.0
sim(good, terrible)   : 0.281
sim(good, shoe)       : 0.163


1.4 REal sentense embeddings with 'sentense-transformer'

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")         # small, fast, 384-dim -- great default for RAG

docs = [
    "AlfredX is a PyQt5 based AI voice assistant.",
    "AlfredX supports voice commands and reminders.",
    "The stock market fell sharply today.",
]
doc_embeddings = model.encode(docs)             # shape: (3, 384)
query_embedding = model.encode("Tell me about AI campanion app")

from sklearn.metrics.pairwise import  cosine_similarity
sims = cosine_similarity([query_embedding], doc_embeddings)[0]
for doc, score in sorted(zip(doc, sims), key= lambda x : -x[1]):
    print(round(score, 3), doc)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.276 l
0.273 A
0.007 f


2. chunking

In [15]:
sample_doc = '''
AlfredX is a PyQt5-based AI voice assistant built by a four-member team. It uses the Groq API
with Llama 3 for language understanding and supports over one hundred voice and text commands.

The core AI engine was rewritten to support full multi-turn conversation history, streaming
responses from Groq, and rate-limit retry logic. Seven critical bugs were fixed during
development, including a duplicate class definition and broken context handling.

The interface is bilingual, supporting English and Turkish, with a JARVIS-style HUD featuring
animated system monitors and a voice waveform visualization. A FastAPI and WebSocket based
mobile companion app allows remote control of the assistant from a phone.
'''.strip()
print(len(sample_doc), "characters")

709 characters


2.1 Simplest possible: fixed size character chunks(naive cuts words/sentence in half)

In [ ]:
def naive_chunk(text, chunk_size):
    return [text[i:i+chunks_size]] for in in range(0, len(text), chunk_size)


chunks = naive_chunk(sample_doc, chunk_size=100)
for i, c in enumerate(chunks):
        print(f"--- chunk {i},")
        print(repr(c))
        

2.2 Fixed sixe with overlap

2.3 Better: split on sentence/paragraph boundries first

2.4 Production grade: